# 07 — Customer lifetime value, and a model that does not fit

3.03% of Olist's customers ever bought twice, and those who did averaged 1.11
repeat purchases. This notebook establishes what follows from that, which is more
interesting than a fitted curve would have been.

In [ ]:
import json
import warnings

import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [ ]:
try:
    clv = read_metric("clv", METRICS)
except FileNotFoundError:
    clv = None
    print("metrics/clv.json not present; run `make clv`.")

if clv:
    print(json.dumps(clv["repeat_behaviour"], indent=2))

## Maximum likelihood does not converge, and it is not a tuning problem

BG/NBD is the standard model and `lifetimes` is the reference implementation. It
does not converge here — at any time scale, at any penalty, on the full base or on
the repeaters alone. The likelihood returns NaN and the parameters run off in log
space.

The reason is structural. BG/NBD's dropout parameters describe the shape of a Beta
distribution over the probability of churning after each purchase, and they are
identified only by the *pattern* of repeat purchasing. With repeaters averaging
1.11 repeats there is no pattern for them to be estimated from, so the likelihood
is flat in those directions.

In [ ]:
if clv:
    ml = clv["maximum_likelihood"]
    show(pd.DataFrame(ml["attempts_full_base"])[["time_unit", "penalizer", "converged"]],
         "Full base")
    show(pd.DataFrame(ml["attempts_repeaters_only"])[["time_unit", "penalizer", "converged"]],
         "Repeaters only")
    print(ml["finding"])

## The Bayesian fit converges because its priors do the work

This is not the Bayesian model being better. pymc-marketing parameterises dropout
with a Pareto prior whose shape parameter is one, which has no finite mean — an
extremely diffuse prior. If the posterior on those parameters barely narrows
against it, the data has told us nothing about the dropout process, which is
exactly what maximum likelihood was unable to estimate.

The ratio of posterior to prior standard deviation is reported per parameter so
that can be read rather than assumed.

In [ ]:
if clv:
    parameters = clv["models"]["bgnbd_bayesian_full_base"]["parameters"]
    print("declared priors:")
    for name, prior in parameters.get("declared_priors", {}).items():
        print(f"  {name:16s} {prior}")
    print()
    rows = [{"parameter": name, **{k: round(v, 4) if isinstance(v, float) else v
                                    for k, v in entry.items()}}
            for name, entry in parameters.items() if name != "declared_priors"]
    if rows:
        show(pd.DataFrame(rows))
    print(clv["models"]["bgnbd_bayesian_full_base"]["note"])

## Validation on a time-based holdout

Never a random split: the prediction is "how many purchases in the next N weeks",
and a random split would let the model see the future of the customers it is
forecasting.

The comparison that matters is against a baseline that predicts nothing. A
near-zero prediction matching a near-zero outcome is the model working; it is only
*skill* if it beats predicting zero.

In [ ]:
if clv:
    validation = {k: v for k, v in clv["validation"].items() if not isinstance(v, str)}
    for key, value in validation.items():
        print(f"{key:44s} {value}")
    print()
    print(clv["validation"]["metric_note"])

## What lifetime value comes to, and why it decides notebook 08

If lifetime value is very nearly proportional to first-order value, then weighting
a media allocation by lifetime value and weighting it by immediate revenue rank the
channels identically and produce the same budget. The CLV-weighted reallocation the
brief invites is then not a different answer — it is the same answer with more
steps, and saying so is the finding.

In [ ]:
if clv:
    value = clv["lifetime_value"]
    for key, item in value.items():
        print(f"{key:44s} {item}")
    print()
    print(clv["finding"])